# **任务16 Positional Encoding | 位置编码**

位置编码是一种赋予特征位置信息的思想，本质上是为其赋予无关已有特征的外部参考信息的操作，这种操作同样有不同的方法，这里展示两种。

它的出现是为了解决线性层不能理解空间信息的，至于为什么非要让它理解空间信息呢？我们不是已经有卷积和循环神经网络了吗？卷积的感受野有限，不能理解较远的依赖关系，RNN则是因为其循环结构，单层网络需要先后计算多次才能输入一整个句子，而线性层的结构允许它并行计算一整个句子。

## 1. **位置索引嵌入**

给每个位置生成一个特殊的嵌入张量，并按照索引顺序，每次都融合相同的嵌入特征。

In [34]:
import torch
import torch.nn as nn

class PosEncoding(nn.Module):
    def __init__(self, seq_num, embedding_dim):
        super().__init__()
        self.pos_embedding = nn.Embedding(num_embeddings=seq_num, embedding_dim=embedding_dim)

    def forward(self, x):
        batch, seq_num, embedding_dim = x.shape
        idx_tensor = torch.arange(seq_num).unsqueeze(0).repeat(batch, 1)
        pos_encoding = self.pos_embedding(idx_tensor)
        print('- 位置编码层输入：', x.shape)
        print('- 位置编码：', pos_encoding.shape)
        x += pos_encoding
        return x

### 1.1 **定义模型**

位置编码是为了解决线性层会忽略空间特征所创造的，所以这里可以直接使用线性层。

In [35]:
class MainModel(nn.Module):
    def __init__(self, seq_num, num_embeddings, embedding_dim, output_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.pos_encoding = PosEncoding(seq_num=seq_num, embedding_dim=embedding_dim)

        self.layer = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, output_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        print('- 添加位置编码后的张量：', x.shape)
        x = self.layer(x)
        return x

### 1.2. **模拟前向传播**

In [36]:
import random

embedding_dim = 16
batch_size = 2
seq_num = 16

vocab_dict = {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
vocab_size = len(vocab_dict)

batch_list = []
for _ in range(batch_size):
    # 随机生成一个长度为seq_num的句子
    sentence = random.choices(list(vocab_dict.keys()), k=seq_num)
    # 转换成索引
    sentence_index = [vocab_dict[char] for char in sentence]
    # 转换成张量
    sentence_tensor = torch.tensor(sentence_index)
    # 添加批次维度
    sentence_tensor = sentence_tensor.unsqueeze(dim=0)
    batch_list.append(sentence_tensor)

# 拼接成一整个批次
input_tensor = torch.cat(batch_list, dim=0)

model = MainModel(seq_num, num_embeddings=vocab_size, embedding_dim=embedding_dim, output_dim=8, hidden_dim=16)

output_tensor = model(input_tensor)
print(f'输入张量：{input_tensor.shape}')
print(f'输出张量：{output_tensor.shape}')

- 位置编码层输入： torch.Size([2, 16, 16])
- 位置编码： torch.Size([2, 16, 16])
- 添加位置编码后的张量： torch.Size([2, 16, 16])
输入张量：torch.Size([2, 16])
输出张量：torch.Size([2, 16, 8])


位置编码是直接加上去的，实际上可以有不同的处理方法，添加位置编码前后的张量形状不变。

## 2. **正余弦位置编码**

它并不能自适应学习，但已经可以基本完成位置编码的需求。并且相较于嵌入计算量极低，适合大规模数据的位置编码嵌入，比如图像和长文本。

In [37]:
import torch
import torch.nn as nn

class SinCosPosEncoding(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim  # 仅需记录维度，无需初始化参数

    def forward(self, x):
        # 获取输入形状：[batch, seq_num, embedding_dim]
        batch, seq_num, embedding_dim = x.shape

        # 1. 生成位置索引 [seq_num]
        pos = torch.arange(seq_num, device=x.device)
        # 2. 生成维度索引（步长2，对应偶数维度）[embedding_dim//2]
        dim_idx = torch.arange(0, embedding_dim, 2, device=x.device)

        # 3. 计算正弦余弦的分母项：10000^(2i/d_model)
        denom = torch.pow(10000, 2 * dim_idx / self.embedding_dim)

        # 4. 初始化位置编码张量 [seq_num, embedding_dim]
        pos_encoding = torch.zeros(seq_num, embedding_dim, device=x.device)
        # 5. 填充偶数维度（sin）、奇数维度（cos）
        pos_encoding[:, 0::2] = torch.sin(pos.unsqueeze(1) / denom)  # 偶数位：sin
        pos_encoding[:, 1::2] = torch.cos(pos.unsqueeze(1) / denom)  # 奇数位：cos

        print('- 位置编码层输入：', x.shape)
        print('- 位置编码：', pos_encoding.shape)
        # 6. 扩展batch维度 [1, seq_num, embedding_dim] → [batch, seq_num, embedding_dim]
        pos_encoding = pos_encoding.unsqueeze(0).repeat(batch, 1, 1)
        # 7. 叠加位置编码（保持输入输出形状一致）
        x += pos_encoding
        return x

### 2.1 **定义模型**

In [38]:
class MainModel(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, output_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.pos_encoding = SinCosPosEncoding(embedding_dim=embedding_dim)

        self.layer = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, output_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        print('- 添加位置编码后的张量：', x.shape)
        x = self.layer(x)
        return x

### 2.2. **模拟前向传播**

In [39]:
embedding_dim = 16
batch_size = 2
seq_num = 16

vocab_dict = {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
vocab_size = len(vocab_dict)

batch_list = []
for _ in range(batch_size):
    # 随机生成一个长度为seq_num的句子
    sentence = random.choices(list(vocab_dict.keys()), k=seq_num)
    # 转换成索引
    sentence_index = [vocab_dict[char] for char in sentence]
    # 转换成张量
    sentence_tensor = torch.tensor(sentence_index)
    # 添加批次维度
    sentence_tensor = sentence_tensor.unsqueeze(dim=0)
    batch_list.append(sentence_tensor)

# 拼接成一整个批次
input_tensor = torch.cat(batch_list, dim=0)

model = MainModel(num_embeddings=vocab_size, embedding_dim=embedding_dim, output_dim=8, hidden_dim=16)

output_tensor = model(input_tensor)
print(f'输入张量：{input_tensor.shape}')
print(f'输出张量：{output_tensor.shape}')

- 位置编码层输入： torch.Size([2, 16, 16])
- 位置编码： torch.Size([16, 16])
- 添加位置编码后的张量： torch.Size([2, 16, 16])
输入张量：torch.Size([2, 16])
输出张量：torch.Size([2, 16, 8])


这里的位置编码与嵌入存在最明显的差异：

嵌入位置编码的形状是 `[batch, seq, emb]`，而正余弦位置编码生成的实际上只有 `[batch, seq]` 两个维度.

这里就可以使用两种方式将其融合，第一种：扩展维度到 `[batch, seq, 1]` 然后使用广播乘法让其与每个 `emb` 维度融合；第二种也就是当前使用的，就是使用 `pos_encoding.unsqueeze(0).repeat(batch, 1, 1)` 将其扩展到 `[batch, seq, emb]` 然后与输入加和计算。

## 3. **总结**

循环神经网络和卷积虽然能理解空间信息，但它们各自都有致命且难以解决的缺点，为了使用线性层替代它们，我们使用位置编码来解决线性层不能理解空间信息的问题，这样线性层就同样具备了各自的长处了。这就是Transformer架构的关键之一。